In [ ]:
import os

import matplotlib.pyplot as plt
import onnx
import onnxruntime
import pytorch_lightning as pl
import torch
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from pytorch_lightning.loggers import MLFlowLogger
from torch.utils.data import DataLoader
from torchvision.io import read_image
from torchvision.transforms import v2

from dataset import SegmentationDataset
from model import LeafModel

# Enable Tensor Cores on A100 for ~2x speed boost
torch.set_float32_matmul_precision("medium")

%matplotlib inline
%load_ext autoreload
%autoreload 2

Create datasets and dataloaders

In [ ]:
# We have to resize the images and masks to a fixed size for batching
dataset_path = "datasets/strawberry"

transforms = v2.Compose(
    [
        v2.Resize((256, 256), interpolation=v2.InterpolationMode.NEAREST),
    ]
)

train_dataset = SegmentationDataset(
    root_dir=f"{dataset_path}/train/", transform=transforms, target_transform=transforms
)
val_dataset = SegmentationDataset(root_dir=f"{dataset_path}/valid/", transform=transforms, target_transform=transforms)
test_dataset = SegmentationDataset(root_dir=f"{dataset_path}/test/", transform=transforms, target_transform=transforms)

print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

n_cpu = os.cpu_count() or 8
# Increased batch size from 32 to 64 for fewer iterations and better GPU utilization
train_dataloader = DataLoader(
    train_dataset, batch_size=32, shuffle=True, num_workers=n_cpu - 2, pin_memory=True, persistent_workers=True
)
valid_dataloader = DataLoader(
    val_dataset, batch_size=32, shuffle=False, num_workers=n_cpu - 2, pin_memory=True, persistent_workers=True
)
test_dataloader = DataLoader(
    test_dataset, batch_size=32, shuffle=False, num_workers=n_cpu - 2, pin_memory=True, persistent_workers=True
)

Visualize a few samples from the dataset to ensure that images and masks are loaded correctly.

In [ ]:
num_samples = 2

figure, axes = plt.subplots(num_samples, 2, figsize=(10, 4 * num_samples))
figure.suptitle("Sample Images and Masks from Training Set", fontsize=16, y=0.995)

for i in range(num_samples):
    img_id = torch.randint(0, len(train_dataset), (1,)).item()
    image, mask = train_dataset[img_id]

    # Display image
    axes[i, 0].imshow(image.permute(1, 2, 0))
    axes[i, 0].set_title(f"Image {i + 1}")
    axes[i, 0].axis("off")

    # Display mask
    axes[i, 1].imshow(mask.squeeze(), cmap="gray")
    axes[i, 1].set_title(f"Mask {i + 1}")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()

Enable mlflow logging

In [ ]:
# Create temporary directory for local artifacts
# temp_dir = tempfile.mkdtemp()

os.environ["MLFLOW_TRACKING_TOKEN"] = ""

mlflow_logger = MLFlowLogger(
    experiment_name="leaf_segmentation_tomato",
    tracking_uri="https://gitlab.com/api/v4/projects/72607084/ml/mlflow/",
    log_model=True,
    # save_dir=temp_dir,  # Temporary local storage
)

Configure model

In [ ]:
EPOCHS = 300
T_MAX = EPOCHS * len(train_dataloader)
OUT_CLASSES = 1
model = LeafModel("FPN", "resnet34", in_channels=3, out_classes=1, t_max=T_MAX)

# Optional: Compile model for faster training (PyTorch 2.0+)
# Uncomment if you have PyTorch 2.0 or later
# model = torch.compile(model)

Set hyperparameters and start training loop

In [ ]:
# Early stopping callback - monitors validation loss
early_stop_callback = EarlyStopping(
    monitor="val_loss",
    patience=10,  # Increased patience for 300 epochs
    mode="min",
    verbose=True,
)

# Model checkpoint callback - only saves best model to MLflow
checkpoint_callback = ModelCheckpoint(
    # dirpath=temp_dir,  # Temporary local storage
    filename="best-model",
    monitor="val_loss",
    mode="min",
    save_top_k=1,  # Only keep the best model
    save_last=False,  # Don't save the last model
    every_n_epochs=1,  # Check only at end of each epoch (not every batch)
)

trainer = pl.Trainer(
    max_epochs=EPOCHS,
    logger=mlflow_logger,
    callbacks=[early_stop_callback, checkpoint_callback],
    enable_checkpointing=True,
)

trainer.fit(
    model,
    train_dataloaders=train_dataloader,
    val_dataloaders=valid_dataloader,
)

# Clean up temporary directory after training
# shutil.rmtree(temp_dir, ignore_errors=True)

Validation Metrics

In [ ]:
valid_metrics = trainer.validate(model, dataloaders=valid_dataloader, verbose=False)
print(valid_metrics)

Visualize predictions

In [ ]:
import random

num_samples = 10
# Get all test samples
all_samples = list(range(len(test_dataset)))
random_indices = random.sample(all_samples, min(num_samples, len(all_samples)))

for idx in random_indices:
    image, gt_mask = test_dataset[idx]

    with torch.inference_mode():
        model.eval()
        logits = model(image.unsqueeze(0))

    pr_mask = logits.sigmoid().squeeze()

    plt.figure(figsize=(10, 5))
    plt.subplot(1, 3, 1)
    plt.imshow(image.permute(1, 2, 0))
    plt.title("Image")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(gt_mask.squeeze(), cmap="gray")
    plt.title("Ground truth")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(pr_mask.detach().numpy(), cmap="gray")
    plt.title("Prediction")
    plt.axis("off")
    plt.show()

Load model from checkpoint

In [ ]:
MODEL_CHECKPOINT_PATH = "best-model.ckpt"

model = LeafModel.load_from_checkpoint(
    MODEL_CHECKPOINT_PATH,
    arch="FPN",
    encoder_name="resnet34",
    in_channels=3,
    out_classes=1,
    t_max=1,  # Dummy value, not used in inference
)

Test model inference

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGE_PATH = "datasets/strawberry/valid/dataset1_2bd8ab67-IMG_9344_jpg.rf.7d67c57277027a9cd34451062fdf2fe6.jpg"

model.to(DEVICE)
model.eval()

image = read_image(IMAGE_PATH).float()  # Keep as uint8 equivalent, no /255.0 to match training

# Apply transforms
transforms = v2.Compose(
    [
        v2.Resize((256, 256), interpolation=v2.InterpolationMode.NEAREST),
    ]
)
image = transforms(image)

# Run inference
print("Running inference...")
with torch.no_grad():
    image_batch = image.unsqueeze(0).to(DEVICE)  # Add batch dimension
    logits = model(image_batch)
    pred_mask = logits.sigmoid().squeeze().cpu()

print(f"Prediction mask range: {pred_mask.min().item():.4f} - {pred_mask.max().item():.4f}")

# Visualize results
plt.figure(figsize=(10, 5))

# Original image
plt.subplot(1, 2, 1)
plt.imshow(image.permute(1, 2, 0) / 255.0)  # Normalize for display
plt.title("Input Image")
plt.axis("off")

# Predicted mask
plt.subplot(1, 2, 2)
plt.imshow(pred_mask.numpy(), cmap="gray")
plt.title("Predicted Mask")
plt.axis("off")
plt.tight_layout()

Convert to ONNX

Create wrapper model with preprocessing for ONNX export

In [ ]:
class ONNXWrapper(torch.nn.Module):
    """Wrapper model that includes preprocessing for ONNX export.

    This wrapper takes raw images (H, W, C) in uint8 format [0-255]
    and handles all preprocessing internally before passing to the model.
    The output is upsampled back to the original input size.
    """

    def __init__(self, model: torch.nn.Module, target_size: tuple[int, int] = (256, 256)) -> None:
        """Initialize the ONNX wrapper.

        Args:
            model: The PyTorch model to wrap.
            target_size: The target size for resizing images (height, width).
        """
        super().__init__()
        self.model = model
        self.target_size = target_size

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass with preprocessing and postprocessing.

        Args:
            x: Input tensor of shape (B, C, H, W) or (B, H, W, C) in range [0-255].

        Returns:
            Output probabilities of shape (B, 1, H, W) in range [0-1], same spatial size as input.
        """
        # x should be (B, H, W, C) or (B, C, H, W) in uint8 range [0-255]

        # If input is (B, H, W, C), convert to (B, C, H, W)
        if x.shape[-1] == 3:
            x = x.permute(0, 3, 1, 2)

        # Store original input size for upsampling output
        original_size = (x.shape[2], x.shape[3])  # (H, W)

        # Ensure float type (still in 0-255 range, NOT normalized to 0-1)
        x = x.float()

        # Resize to target size using bilinear interpolation
        x = torch.nn.functional.interpolate(x, size=self.target_size, mode="bilinear", align_corners=False)

        # Pass through the model (model handles normalization internally)
        logits = self.model(x)

        # Apply sigmoid to get probabilities
        probs = torch.sigmoid(logits)

        # Upsample output back to original input size
        probs = torch.nn.functional.interpolate(probs, size=original_size, mode="bilinear", align_corners=False)

        return probs


# Create wrapper with your loaded model
wrapped_model = ONNXWrapper(model, target_size=(256, 256))
wrapped_model.eval()

print("Wrapper model created successfully!")
print("Input: any size -> downsampled to 256x256 for inference -> upsampled back to input size")

In [ ]:
ONNX_NAME = "leaf_segmentation.onnx"

wrapped_model.eval()

# Example input: (batch_size, channels, height, width) in 0-255 range
# The model will handle resizing internally, so we can use any input size
dummy_input = torch.randint(0, 256, (1, 3, 480, 640), dtype=torch.float32)

dynamic_axes = {0: "batch_size", 2: "height", 3: "width"}
onnx_model = torch.onnx.export(
    wrapped_model,  # Wrapped model with preprocessing
    dummy_input,  # model input (any size, in 0-255 range)
    ONNX_NAME,  # where to save the model
    export_params=True,  # store the trained parameter weights inside the model file
    opset_version=17,  # the ONNX version to export
    do_constant_folding=True,  # whether to execute constant folding for optimization
    input_names=["input"],  # the model's input names
    output_names=["output"],  # the model's output names (probabilities after sigmoid)
    dynamic_axes={  # variable length axes
        "input": dynamic_axes,
        "output": dynamic_axes,  # Output will match input size (B, 1, H, W)
    },
)

print(f"ONNX model exported to {ONNX_NAME}")
print("Input: (batch_size, 3, height, width) - any size, values in [0, 255]")
print("Output: (batch_size, 1, height, width) - same size as input, probabilities in [0, 1]")

In [ ]:
onnx_model = onnx.load(ONNX_NAME)
onnx.checker.check_model(onnx_model)

Log ONNX model to MLflow

In [ ]:
import mlflow

# Log the ONNX model to MLflow
with mlflow.start_run(run_id=mlflow_logger.run_id):
    # Log the ONNX model as an artifact
    mlflow.log_artifact(ONNX_NAME, artifact_path="onnx_model")
    mlflow.log_artifact(f"{ONNX_NAME}.data", artifact_path="onnx_model")

inference using onnx

In [ ]:
# Load image - NO preprocessing needed!
image = read_image(IMAGE_PATH).float()  # Just load and convert to float

# Prepare input for ONNX runtime - just add batch dimension
onnx_input = image.unsqueeze(0).numpy()  # (1, 3, H, W) in [0, 255] range

# Create ONNX Runtime session
ort_session = onnxruntime.InferenceSession(ONNX_NAME)

# Run inference - the model handles resizing and normalization internally!
ort_inputs = {ort_session.get_inputs()[0].name: onnx_input}
ort_output = ort_session.run(None, ort_inputs)[0]

# Output is already probabilities (sigmoid applied), just squeeze
onnx_prediction = torch.from_numpy(ort_output).squeeze()

# Visualize results
plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.imshow(image.permute(1, 2, 0).numpy() / 255.0)  # Normalize for display only
plt.title(f"Input Image ({image.shape[1]}x{image.shape[2]})")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(onnx_prediction.numpy(), cmap="gray")
plt.title(f"ONNX Prediction ({onnx_prediction.shape[0]}x{onnx_prediction.shape[1]})")
plt.axis("off")

plt.tight_layout()
plt.show()

print(f"Input shape: {onnx_input.shape}")
print(f"Output shape: {ort_output.shape}")
print(f"Output matches input size: {ort_output.shape[2:] == onnx_input.shape[2:]}")
print(f"Output range: [{onnx_prediction.min():.4f}, {onnx_prediction.max():.4f}]")

Test with different image sizes (no manual preprocessing needed!)

In [ ]:
# Test with multiple images of different sizes
test_images = [
    "datasets/strawberry/valid/dataset2_771_png.rf.64f0abd171d9fde7559a7b3b9ce929da.jpg",
    "datasets/strawberry/valid/IMG_6399_jpg.rf.6c7644155bd8ee28db94d10cd9c83791.jpg",
]

ort_session = onnxruntime.InferenceSession(ONNX_NAME)

for img_path in test_images:
    # Just load the image - that's it!
    img = read_image(img_path).float()

    # Run inference
    onnx_input = img.unsqueeze(0).numpy()
    ort_output = ort_session.run(None, {ort_session.get_inputs()[0].name: onnx_input})[0]
    prediction = torch.from_numpy(ort_output).squeeze()

    # Display
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(img.permute(1, 2, 0).numpy() / 255.0)
    plt.title(f"Input: {img.shape[1]}x{img.shape[2]}")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(prediction.numpy(), cmap="gray")
    plt.title(f"Prediction: {prediction.shape[0]}x{prediction.shape[1]} (matches input!)")
    plt.axis("off")

    plt.tight_layout()
    plt.show()